<a href="https://colab.research.google.com/github/grfaith/AmericanStories/blob/main/Get_AS_text_and_light_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ipympl
!pip install symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.7/515.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 13.1 MB/s eta 0:00:00


In [2]:
# ── 1. Mount your Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── 2. Imports ─────────────────────────────────────────────────────────────
import os
import glob
import pandas as pd


Mounted at /content/drive


In [4]:
# ── Cell: Post-processing definitions ──────────────────────────────────────
import pkg_resources
from symspellpy import SymSpell, Verbosity
import string

# 1. Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dict_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dict_path, term_index=0, count_index=1)

# 2. Merge hyphenated or split lines
def line_merge(text):
    lines = [l.split() for l in text.split("\n")]
    for i in range(len(lines) - 1):
        if not lines[i] or not lines[i+1]:
            continue
        # if ends with a hyphen, join across the break
        if lines[i][-1].endswith("-"):
            lines[i][-1] = lines[i][-1][:-1] + lines[i+1][0]
            lines[i+1] = lines[i+1][1:]
        else:
            a = lines[i][-1].strip(string.punctuation).lower()
            b = lines[i+1][0].strip(string.punctuation).lower()
            # if the concatenation is a known word, merge
            if (a + b) in sym_spell.words:
                lines[i][-1] += lines[i+1][0]
                lines[i+1] = lines[i+1][1:]
    return "\n".join(" ".join(l) for l in lines)

# 3. Spell-check helper
def check_word(word):
    core = word.strip(string.punctuation)
    if not core:
        return word
    suggestions = sym_spell.lookup(
        core,
        Verbosity.CLOSEST,
        max_edit_distance=1,
        include_unknown=True,
        transfer_casing=True
    )
    return word.replace(core, suggestions[0].term)

def spell_check(text):
    return "\n".join(
        " ".join(check_word(w) for w in line.split(" "))
        for line in text.split("\n")
    )

# 4. Capitalization fixer
def capitalization_check(text):
    out_lines = []
    for line in text.split("\n"):
        words = line.split(" ")
        for i in range(1, len(words)):
            if words[i-1].endswith((".", "!", "?")):
                words[i] = words[i].capitalize()
            else:
                core = words[i].strip(string.punctuation).lower()
                if core in sym_spell.words and core not in ("i", "i'll"):
                    words[i] = words[i].replace(core, core.lower())
        out_lines.append(" ".join(words))
    return "\n".join(out_lines)

# 5. Full pipeline
def postprocess(text):
    merged = line_merge(text)
    checked = spell_check(merged)
    return capitalization_check(checked)


In [5]:
import os
import json
import tarfile
import requests
import pandas as pd
from tqdm import tqdm
from pandas.errors import EmptyDataError

# ── CONFIG ─────────────────────────────────────────────────────────────
GROUPED_DIR = '/content/drive/MyDrive/AmStories_grouped'
OUT_DIR     = '/content/drive/MyDrive/AmStories_text'
CACHE_DIR   = '/tmp/americanstories'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── HELPERS ────────────────────────────────────────────────────────────
def json_filename_from_aid(aid: str) -> str:
    """From '17_1798-11-30_p1_…' → '1798-11-30_p1_….json'."""
    return aid.split('_', 1)[1] + '.json'

def json_id_from_aid(aid: str) -> str:
    """From '17_1798-11-30_p1_…' → '17_1798-11-30_p1_….json'."""
    return aid + '.json'

def fetch_and_open_tar(year: int) -> tarfile.TarFile:
    """Download (if needed) and open the year’s tarball."""
    tar_name = f'faro_{year}.tar.gz'
    tar_url  = (
        'https://huggingface.co/datasets/'
        'dell-research-harvard/AmericanStories/resolve/main/'
        + tar_name
    )
    local_tar = os.path.join(CACHE_DIR, tar_name)
    if not os.path.exists(local_tar):
        print(f"↓ Downloading {tar_name}…")
        resp = requests.get(tar_url, stream=True)
        resp.raise_for_status()
        with open(local_tar, 'wb') as out:
            for chunk in resp.iter_content(8_192):
                out.write(chunk)
    return tarfile.open(local_tar, 'r:gz')

def build_member_index(tf: tarfile.TarFile) -> dict:
    """Map JSON filename → TarInfo for quick lookup."""
    idx = {}
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith('.json'):
            fname = os.path.basename(m.name)
            idx[fname] = m
    return idx

def extract_fragment(tf: tarfile.TarFile,
                     member_idx: dict,
                     aid: str) -> tuple[str|None, dict|None]:
    """
    Given an ARTICLE_ID, open its JSON, and find the art entry
    whose id exactly matches aid+'.json'. Returns (text, bbox) or (None, None).
    """
    jfn = json_filename_from_aid(aid)
    member = member_idx.get(jfn)
    if not member:
        return None, None

    with tf.extractfile(member) as f:
        page = json.load(f)

    target_id = json_id_from_aid(aid)
    art = next(
        (a for a in page.get('full articles', [])
         if a.get('id') == target_id),
        None
    )
    if art is None:
        return None, None

    # assemble text
    parts = []
    for fld in ('headline', 'byline', 'article'):
        t = art.get(fld)
        if t:
            parts.append(t.strip())
    combined_text = "\n\n".join(parts)

    # use only 'bbox'
    bbox = art.get('bbox')
    return combined_text, bbox

def make_article_link(aid: str, bbox: list|None) -> str|None:
    """
    Build a LOC clip URL:
      https://www.loc.gov/resource/{sn}/{date}/ed-1/?sp={page}&clip={x0},{y0},{x1},{y1}
    from article_ID and bbox.
    """
    if not bbox:
        return None
    parts = aid.split('_')
    date = parts[1]
    page = parts[2].lstrip('p')
    sn   = parts[3]
    clip = ','.join(str(v) for v in bbox)
    return f"https://www.loc.gov/resource/{sn}/{date}/ed-1/?sp={page}&clip={clip}"

# ── MAIN LOOP ──────────────────────────────────────────────────────────
for year in range(1773, 1801):
    csv_fp = os.path.join(GROUPED_DIR, f'Grouped_KW_Hits_May25_SW_{year}.csv')
    if not os.path.isfile(csv_fp):
        continue

    try:
        df = pd.read_csv(csv_fp, converters={'keyword_counts': eval})
    except EmptyDataError:
        print(f"⚠️  Year {year}: empty CSV, skipping.")
        continue
    if df.empty:
        print(f"⚠️  Year {year}: no rows, skipping.")
        continue

    # open tar & index JSON members
    tf         = fetch_and_open_tar(year)
    member_idx = build_member_index(tf)
    print(f"🔍 Year {year}: indexed {len(member_idx)} JSON files.")

    # extract text & bbox
    df['analyze_text'] = df['article_ID'].map(
        lambda aid: (lambda t, _b: postprocess(t) if t else None)(*extract_fragment(tf, member_idx, aid))
    )
    df['bbox'] = df['article_ID'].map(
        lambda aid: extract_fragment(tf, member_idx, aid)[1]
    )

    # build the clip URL
    df['article_link'] = df.apply(
        lambda row: make_article_link(row['article_ID'], row['bbox']),
        axis=1
    )

    # save
    out_fn = os.path.join(OUT_DIR, f'AS_Text_Analyzed_May25_SW_{year}.csv')
    df.to_csv(out_fn, index=False)
    print(f"✔ Year {year} done. Wrote {out_fn}.")

    tf.close()

print("ALL YEARS COMPLETE!")


🔍 Year 1774: indexed 176 JSON files.
✔ Year 1774 done. Wrote /content/drive/MyDrive/AmStories_text/AS_Text_Analyzed_May25_SW_1774.csv.
⚠️  Year 1777: empty CSV, skipping.
⚠️  Year 1778: empty CSV, skipping.
⚠️  Year 1779: empty CSV, skipping.
⚠️  Year 1791: empty CSV, skipping.
⚠️  Year 1792: empty CSV, skipping.
⚠️  Year 1793: empty CSV, skipping.
⚠️  Year 1796: empty CSV, skipping.
⚠️  Year 1797: empty CSV, skipping.
🔍 Year 1798: indexed 208 JSON files.
✔ Year 1798 done. Wrote /content/drive/MyDrive/AmStories_text/AS_Text_Analyzed_May25_SW_1798.csv.
↓ Downloading faro_1799.tar.gz…
🔍 Year 1799: indexed 201 JSON files.
✔ Year 1799 done. Wrote /content/drive/MyDrive/AmStories_text/AS_Text_Analyzed_May25_SW_1799.csv.
↓ Downloading faro_1800.tar.gz…
🔍 Year 1800: indexed 348 JSON files.
✔ Year 1800 done. Wrote /content/drive/MyDrive/AmStories_text/AS_Text_Analyzed_May25_SW_1800.csv.
ALL YEARS COMPLETE!
